# BB84 Protocol Simulator

## Introduction

BB84 is the first and most widely used Quantum Key Distribution (QKD) scheme developed by Bennet and Brassard in 1984.  
It utilizes the *no-cloning theorem*, a law of quantum mechanics, that let two parties share a random key securely over an unsecure channel, that can be used for secure communication between them.  
The main advantage that BB84 provides is that eavesdroppers are detected if the error-rate is above a certain threshold.  

## The No-Cloning Theorem

A fundamental principle of quantum mechanics that states that it is impossible to create a copy of an arbitrary quantum state.  

The only way to create a copy of a quantum state is to know the preparation algorithm for given state. That algorithm can be executed as many time to create as many copies of the said quantum state.  

It's used in BB84 to *detect* eavesdroppers (Eve) intercepting the information.  
When Eve tries to measure the qubits, the qubits collapse into a single state. Without knowing how the qubit was prepared, Eve will somehow try to regenerate the qubit. The probability that she will regenerate the same qubit as received by Alice and that Bob will measure it to get the same bit as intended by Alice is very low. 

## The Protocol

Say Alice and Bob need to exchange a key securely over an insecure channel.  

Step 1: Encoding  
Alice generates and sends to Bob, a random sequence of bits and encodes them into photon polarization states having two different sets of orientation (or bases):  
* Rectilinear basis: 0/90 degree basis that may be represented by $|0\rangle and |1\rangle$
* Diagonal basis: 45/135 degree basis that may be represented by $|+\rangle and |-\rangle$  

![The two types of Bases](bases.png)  

Step 2: Measuring  
Bob receives the photons and randomly chooses a basis to measure each of the received photon.  

Step 3: Sifting  
Alice and Bob communicate over a classical, unsecure channel and compare the basis they each chose for each photon, without revealing the bits. They discard the instances where their chosen basis do not match, and the rest of the bits become the key.  

Step 4: Error detection  
Alice and Bob sacrifice a part of their key and compare them over the unsecure classical channel. They calculate error rates and if it's >11%, they discard the key entirely and know that a spy was intercepting the channel.

## Implementation in Qiskit

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [2]:
def alice_prepare(n:int)->(np.ndarray,np.ndarray):
    bits = np.random.randint(2, size=n) #0 and 1 correspond to equivalent bit value
    bases = np.random.randint(2, size=n) #0 and 1 correspond to 0/90 (combination of | and -)(Rectilinear basis |0> and |1>) and 45/135 (combination of \ and /)(Diagonal basis |+> and |->)
    return bits, bases

In [3]:
def bob_choose(n:int)->np.ndarray:
    bases = np.random.randint(2, size=n)
    return bases

In [4]:
def eve_intercept(i:int, qc:QuantumCircuit, eve_basis:np.ndarray) -> QuantumCircuit:
    if eve_basis[i]==1:
        qc.h(0)
        
    qc.measure(0,0)
    simulator = AerSimulator()
    result = simulator.run(qc, shots=1, memory=True).result()
    measured = int(result.get_memory()[0])

    new_qc = QuantumCircuit(1,1)
    if(measured==1):
        new_qc.x(0)
    return new_qc

In [5]:
def encode_qubits(n:int, alice_bits:np.ndarray, alice_bases:np.ndarray, bob_bases:np.ndarray, eve_present:bool) -> list[int]:
    bob_bit_result = []
    simulator = AerSimulator()
    
    if eve_present:
        eve_basis = np.random.randint(2, size=n)
        
    for i in range(n):
        
        qc = QuantumCircuit(1,1)

        if alice_bits[i] == 1:
            qc.x(0)
        if alice_bases[i] == 1:
            qc.h(0)
            
        if eve_present:
            qc = eve_intercept(i, qc, eve_basis)
        
        if bob_bases[i] == 1:
            qc.h(0)

        qc.measure(0,0)

        result = simulator.run(qc, shots=1, memory=True).result()
        measured = int(result.get_memory()[0])
        bob_bit_result.append(measured)
        
    return bob_bit_result

In [6]:
def sift_key(alice_bases:np.ndarray, bob_bases:np.ndarray, alice_bits:np.ndarray, bob_bits:np.ndarray) -> (int,int):
    match = (alice_bases==bob_bases)
    alice_key = alice_bits[match]
    bob_key = bob_bits[match]
    return (alice_key, bob_key)

In [7]:
def calculate_error(alice_key:np.ndarray, bob_key:np.ndarray) -> None:
    error_rate = np.sum(alice_key!=bob_key)/len(alice_key)
    print(f"Error rate: {error_rate:.2%}")
    if(error_rate>0.11):
        print("Eve detected")
    else:
        print("Key exchanged securely")

In [8]:
def bb84(n:int, eve:bool) -> None:
    a_bit, a_base = alice_prepare(n)
    b_base = bob_choose(n)
    b_bit = np.array(encode_qubits(n, a_bit, a_base, b_base, eve_present=True if eve else False))
    a_key, b_key = sift_key(a_base, b_base, a_bit, b_bit)
    print(f"Alice Key: {a_key}\nBob Key: {b_key}")
    calculate_error(a_key, b_key)

## Results

In [9]:
n = int(input("Enter number of bits: "))
eve = True if input("Eve present or no? (y/n): ").lower()=='y' else False

Enter number of bits:  25
Eve present or no? (y/n):  y


In [10]:
bb84(n, eve)

Alice Key: [0 0 0 1 0 1 1 1 0 0 0 0 1 1]
Bob Key: [0 1 0 0 1 1 1 0 0 1 0 0 0 0]
Error rate: 50.00%
Eve detected


In [11]:
n = int(input("Enter number of bits: "))
eve = True if input("Eve present or no? (y/n): ").lower()=='y' else False

Enter number of bits:  20
Eve present or no? (y/n):  n


In [12]:
bb84(n, eve)

Alice Key: [1 0 0 0 0 0 0 1 0 0 0 0]
Bob Key: [1 0 0 0 0 0 0 1 0 0 0 0]
Error rate: 0.00%
Key exchanged securely


## Conclusion

Implementing the BB84 protocol helped me understand the no-cloning theorem concretely. Measurement irreversibly collapses a state in a superposition (a combination of multiple possible states) into a single definite state. And regenerating an unknown quantum state from scratch with only the measurement result has very little probablity. The protocol's strength lies entirely in this.  

However, I noticed a major flaw while implementing it. Although BB84 successfully **detects** eavesdropping, it **does not prevent** it.  
If Eve never cared about the key but simply wants to stop Alice and Bob from ever agreeing on one, then it can easily be done by constantly intercepting every transmission. This is sabotage. It's a denial of service.  
BB84 protects the confidentiality, but not availability. It matters especially for *time-sensitive communication*.  

This same property has a positive flip side. Even when a key can't be agreed upon, the protocol still tells something valuable: whether the channel is being watched. In that sense, BB84 doubles as a *fuse* for suspect channel, independent of whether a key exchange is the end goal. It doesn't guarantee a key, but it always lets you know if someone's listening.